# 4. Modelado: perfiles de demanda y prioridad de refuerzo

**Punto de la rubrica: b (variable objetivo y modelo)**

**Responsable sugerido:** persona A.

Sigue el blueprint de dos niveles de `CLAUDE.md` (Fase 4). **Este es el ultimo notebook de analisis**: las anomalias ya no tienen notebook aparte, son la seccion 4.7 (el EDA, notebook 02, ya hizo la deteccion exploratoria).

## Nivel 1 - Perfiles de demanda (unidad: estacion)

Variable objetivo: `perfil_demanda` (cluster). Medio para organizar turnos, no el objetivo final.

### 4.1 Preparacion

Cargar `estacion_features.parquet` (notebook 02). Variables que entran: `h00...h23`, `razon_am_pm`, `razon_finde_habil`, `indice_concentracion_pico`. Variables que solo describen el cluster: `demanda_promedio_diaria`, `pct_adulto_mayor`, `pct_anonymous`. `StandardScaler`.

### 4.2 Seleccion de k

K-Means con k = 2...8. Codo + silhouette. Validar con Davies-Bouldin.

### 4.3 Comparacion con jerarquico / DBSCAN (opcional)

Dendrograma; comparar asignaciones.

### 4.4 Interpretacion de clusters

Curva horaria promedio por cluster (con banda de variacion) + nombre de negocio (residencial, destino laboral, portal, baja demanda). Visualizar con PCA 2D.

## Nivel 2 - Prioridad de refuerzo (unidad: estacion x tipo de dia x hora)

Variable objetivo: `indice_prioridad` (0-100) y `requiere_refuerzo` (si/no). Granularidad de dia = `tipo_dia` (habil/sabado/domingo), no dia de la semana: con 2 semanas hay solo 2 observaciones por dia de la semana (ver `CLAUDE.md`).

### 4.5 Variables del nivel 2

`validaciones_promedio`, `ratio_vs_promedio_estacion`, `es_hora_critica` (ya construida en el notebook 02, paso 11), `demanda_zona_700m` (ya construida), `perfil_demanda` (del nivel 1).

### 4.6 Indice de prioridad y `requiere_refuerzo`

Combinar las variables del 4.5 en un puntaje 0-100 y fijar el umbral para la version si/no.

### 4.7 Anomalias (detector reutilizable)

Compara cada estacion-hora contra su valor esperado (mismo `tipo_dia`) y deja la tabla `anomalias` (estacion, fecha, hora, observado vs. esperado). Se escribe como funcion en `src/` para que Prefect la corra solo sobre el dia nuevo. Primer caso a revisar: 16-sep, 22-23h (atipico real encontrado en el notebook 02, paso 9). El festivo no debe salir como anomalia (se controla con `tipo_dia`).

### 4.8 Tabla final de turnos

Salida: *"Portal Americas: refuerzo de lunes a viernes de 5 a 8 a.m. (perfil residencial)"* -- una fila por estacion y franja, con su perfil, indice e intervencion sugerida.